https://www.youtube.com/watch?v=DU8o-OTeoCc

# Apache Kafka

Apache Kafka is a distributed event streaming platform used for high-performance data pipelines, streaming analytics, data integration, and mission-critical applications. It is a **publish-subscribe messaging queue**.    

Kafka 最初由 LinkedIn 开发，后捐献给 Apache，是一个分布式的、基于 publish-subscribe 模式的 **消息队列（Message Queue）** 和 **事件流平台（Event Streaming Platform）**。

典型使用场景包括：日志收集（Log Aggregation）、消息系统（Messaging）、用户活动跟踪（Activity Tracking）、运营指标（Operational Metrics）、流式处理（Stream Processing）。

<img src='./pic/4_kafka_usecases.gif' width=500>



## Core Concepts

| 概念 | 说明 |
|---|---|
| **Producer** | 消息生产者，向 Kafka broker 发送消息 |
| **Consumer** | 消息消费者，从 Kafka broker 拉取消息 |
| **Broker** | 一台 Kafka 服务器就是一个 broker，集群由多个 broker 组成 |
| **Topic** | 消息的逻辑分类，类似于数据库中的表 |
| **Partition** | Topic 的物理分片，每个 partition 是一个有序的 commit log |
| **Offset** | 每条消息在 partition 中的唯一递增 ID |
| **Consumer Group** | 消费者组，实现消息的 broadcast（广播）和 unicast（单播）语义 |
| **Replication** | 副本机制，每个 partition 可配置多个 replica 以实现高可用 |
| **Leader / Follower** | 每个 partition 有一个 leader 负责读写，follower 同步数据 |
| **ISR (In-Sync Replicas)** | 与 leader 保持同步的副本集合 |


<img src='./pic/4_Kafka-Architecture.webp' width=600> 

### message sequence
Kafka 只保证**单个 partition 内的消息有序**，不保证跨 partition 的全局顺序。如需全局有序，需将 topic 的 partition 数设为 1（会牺牲吞吐量）。

## ZooKeeper
- Manages **cluster metadata, leader election, and broker coordination**. 
- Tracks which brokers are alive and maintains topic/partition assignments.

## Brokers

- Brokers are Kafka **servers** that **store data and serve clients**. 
- it's a physical or virtual machine running the Kafka process. Think of it as a powerful storage and routing server.
- A Kafka cluster consists of one or more brokers. 
- Brokers **handle read and write requests** from clients and **manage data replication**.


### 4 core management components inside the Broker architecture
```text
Kafka Broker
 ├── Controller        (cluster control)
 ├── Coordinator       (consumer groups)
 ├── Replica Manager   (replication)
 └── Log Manager       (storage)
```

#### **1. Controller（控制器）**       

Controller 是 Kafka 集群中的核心管理节点，负责：
- Partition 的 leader election 
- metadata 管理
- Topic 的创建/删除
- Replica 分配
- Broker 上下线管理    

**Kafka 4.0**（2025年3月发布）已**完全移除 ZooKeeper**。Controller 现在通过 **KRaft（Kafka Raft）mode** 运行，使用内置的 Raft consensus protocol 管理 metadata。Controller 节点组成一个 quorum，通过 replicated metadata log 进行选举和状态管理。

> ⚠️ 过去 Controller 依赖 ZooKeeper 进行选举和状态存储， 现在KRaft model 

#### **2. Coordinator（协调器）**      

- 分为两种：
  - **Group Coordinator**: 管理 Consumer Group 的 member 列表，处理 consumer 加入/离开，触发 rebalance，管理 offset 提交（存储在内部 topic `__consumer_offsets`）。
  - **Transaction Coordinator**: 管理 transactional producer 的事务状态。 
    
Kafka 4.0 GA 了 **KIP-848 新一代 Consumer Group Protocol**，采用 server-side assignment 模式，消除了全局同步屏障，大幅降低 rebalance 对消费的影响。同时引入了 **Incremental Cooperative Rebalancing**（Kafka 2.4+ 已引入，4.0 中成为默认行为），consumer 不再需要在 rebalance 时放弃所有 partition。  
> ⚠️ **过时内容**: rebalance 使用 "stop-the-world" 式的 Eager Rebalance Protocol。

#### **3. Replica Manager（副本管理器）**     

负责管理 partition 的副本同步，关键机制包括：

- **ISR（In-Sync Replicas）**: 与 leader 保持同步的副本集合, 保证数据一致性
- **HW（High Watermark）**: consumer 可见的最大 offset
- **LEO（Log End Offset）**: 每个 replica 最新写入的 offset
- **Leader Epoch**: 用于解决 replica 恢复时的数据一致性问题

Kafka 4.0 引入了 **KIP-966 Eligible Leader Replicas (ELR)**。KRaft controller 现在会追踪不在 ISR 中但可以安全选为 leader 的 replica，存储在 partition metadata 中。这进一步提升了 leader 选举的可靠性，减少了 data loss 风险。

#### **4. Log Manager（日志管理器）**

负责 Kafka 的 commit log 存储和管理：

- 每个 partition 对应磁盘上的一个目录
- 日志按 **Segment** 切分，每个 segment 包含 `.log`（消息数据）、`.index`（offset 索引）、`.timeindex`（时间戳索引）
- 支持 **Log Compaction** 和 **Log Retention**（按时间或大小）
- 使用 **零拷贝（Zero-Copy / sendfile）** 技术提升 I/O 性能
- Log Manager 如何删除数据？
  - retention policy
  - log compaction

Kafka 3.6+ 引入了 **Tiered Storage（分层存储）**，允许将冷数据从本地磁盘迁移到远程存储（如 S3），实现计算和存储的独立扩展。

<img src='./pic/4_new_kafka_architecture.png' width=450>
  
这是基于 Kafka 4.0（KRaft 模式） 的完整架构图，从上到下分四层：
- Producer Layer — 消息经过 Serializer → Partitioner → RecordAccumulator → Sender 线程（NIO），发送到 broker。
- Broker Cluster — 这是最核心的变化区域：
  - KRaft Controller Quorum 取代了原来的 ZooKeeper，通过 Raft 共识协议管理所有 metadata，包括 leader election、topic/partition 管理、broker 注册，以及 4.0 新增的 ELR（Eligible Leader Replicas）追踪
  - 每个 Broker 内部包含：Log Manager（日志管理，segment 存储）、Replica Manager / ISR（副本同步）、Group Coordinator（消费者组管理）、Network Handler（请求处理），以及 3.6+ 新增的 Tiered Storage（灰色，冷数据卸载到 S3）
  - Broker 之间通过 Replica Manager 进行 ISR 同步（绿色箭头）

- __consumer_offsets — 内部 topic，存储所有 consumer group 的 offset 提交。
- Consumer Layer — 两种消费模式并存：
  - Consumer Group A：采用 KIP-848 新协议，server-side assignment，incremental cooperative rebalance
  - Share Group：4.0 Early Access 的 KIP-932，提供传统 queue 语义，消费不再受限于 partition 数量


### Core Responsibilities of a Broker   
1. **Store Messages (Data Persistence)**       
    Brokers physically store all the messages on disk in a structured way.
    
    ```text
        Broker's File System:
        /var/kafka-logs/
        ├── topic-A-partition-0/
        │   ├── 00000000000000000000.log  (actual messages)
        │   ├── 00000000000000000000.index (offset index)
        │   └── 00000000000000000000.timeindex (timestamp index)
        ├── topic-A-partition-1/
        └── topic-B-partition-0/
    ```
    **Each message** is written to disk as an **append-only log (very fast!)**. Messages are organized by:
    - Topic
    - Partition
    - Offset (sequential ID)
2. **Serve Producer Requests (Writes)**       
    When a producer sends a message: 
    
    ```python
    producer.send('my-topic', key=b'user-123', value=b'data')
    ```
    
    The broker:
    - Receives the message
    - Determines which **partition** (based on key or round-robin)
    - **Writes to disk** immediately
    - **Replicates** to follower brokers if it's a leader
    - **Acknowledges back** to producer (based on acks setting)
    ```text
    Producer → [Broker (Leader)] → Write to disk → Replicate → Send ACK
    ```
3. **Serve Consumer Requests (Reads)**       
    When a consumer reads messages:
    
    ```python
    for message in consumer:
    `print(message.value)
    ```
    The broker:
    - Receives fetch request with offset
    - Reads from disk (or page cache for performance)
    - Sends batch of messages back to consumer
    - Tracks consumer position (via consumer group coordinator)
    
    ```text
    Consumer → [Broker (Leader)] → Read from disk/cache → Send messages
    ```
4. **Handle Replication**      
    For **leader brokers**:     
    - Accept writes from producers
    - Push data to follower brokers
    - Track which followers are in-sync (ISR)

    For **follower brokers**:     
    - Continuously fetch data from leader
    - Write to local disk
    - Report sync status back to leader

    Kafka 4.0 introduced **Eligible Leader Replicas(ELR, KIP-966)**. The KRaft controller now tracks replicas that are **NOT in ISR** but are still safe to elect as leader **without data loss**. This provides a middle ground between ISR-only election and unclean leader election, improving availability without sacrificing consistency.

5. **Coordinate with ZooKeeper (out of date)**    
    Brokers communicate with ZooKeeper to:

    - **Register** themselves when they start
    - Report **health status** (heartbeats)
    - Participate in **leader elections**
    - Store **metadata** about topics and partitions
    
    ```text
    Broker ←→ ZooKeeper / Now: KRaft Controller Quorum (Raft consensus)
       ↓
    "I'm alive and managing these partitions"
    ```
    **Coordinate via KRaft (Metadata Management)**
    Starting from Kafka 4.0, ZooKeeper has been completely removed.
    Brokers now coordinate through the built-in **KRaft (Kafka Raft)**
    protocol:

    - **Register** themselves with the KRaft controller quorum
    - Report **health status** via heartbeats to the active controller
    - The controller quorum handles **leader elections** internally
    - All **metadata** (topics, partitions, configurations) is stored
      as a replicated event log managed by the controller quorum

    The KRaft controller quorum consists of dedicated controller nodes
    (or combined broker+controller nodes in smaller deployments) that
    maintain a **metadata log** — applying the same append-only log
    concept Kafka uses for data to its own cluster state.
6. **Manage Consumer Groups**       
    One broker acts as the **group coordinator** for each consumer group:

    - Track which consumers are alive
    - Assign partitions to consumers
    - Handle rebalancing when consumers join/leave
    - Store consumer offsets in `__consumer_offsets` topic   
    
    ```text
    Consumer Group Coordinator (Broker 2):
    - Consumer 1 → Partition 0, 1
    - Consumer 2 → Partition 2, 3
    - Consumer 3 → Partition 4
    ```
    **[2026 Update — KIP-848]**   
    - Kafka 4.0 ships a new **server-side consumer group protocol**. 
    - Assignment logic moves from the client to the group coordinator on the broker, enabling **incremental cooperative rebalancing** 
    - consumers no longer stop consuming all partitions during a rebalance, only the ones being migrated.

    **[2026 Update — KIP-932]** 
    - Kafka 4.0 also introduces **Share Groups** (Early Access), adding queue semantics where multiple consumers can process messages from the same partition concurrently, breaking the traditional 1-partition-per-consumer constraint.

## Topics

- Topics are categories or feed names to which records are published. 
- Topics in Kafka are always **multi-subscriber**, meaning a topic can have zero, one, or many consumers that subscribe to the data written to it.
- Topics are split into **partitions**, which allows for **parallel processing** and **scalability**. 
- Each partition is an **ordered, immutable** sequence of records that is continually appended to.
```text
ONE Topic → MULTIPLE Partitions → DISTRIBUTED across MULTIPLE Brokers
```

**Create Topic explicitly:**   
```bash 
cd kafka-docker 
docker exec -it kafka bash
[appuser@7eb7181d5035 ~]$ kafka-topics --create --topic my-topic --bootstrap-server localhost:9092
```
> will be faster then auto creation

[KIP-1030] Kafka 4.0 changed several **default configuration values for new topics**. For example, `segment.bytes` default is now 1 GB (was 1 GB, unchanged), but the default `replication.factor` when using `auto.create.topics` is now **3** (was 1). 

In [1]:
from kafka import KafkaProducer, KafkaConsumer

# Create a producer instance
producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    # retries=1,      # Reduce initialization wait and retry
    # linger_ms=10,   # Reduce the time waiting for buffered sending
    )
                # bootstrap.servers: Address list of Kafka brokers
                # localhost:9092: Local Kafka server on port 9092
                # Can configure multiple brokers: 'localhost:9092,localhost:9093'
# Send a message to Kafka
producer.send('my-topic', b'Hello World')
            # 'my-topic': Topic name
                # If the topic doesn't exist, Kafka will auto-create it (if broker allows)
            # b'Hello World': Message content
                # b'' indicates bytes type (Kafka transmits binary data)
                # Can also be a string, but needs serialization
# Important: produce() is asynchronous - doesn't send immediately
# Force send all buffered messages
producer.flush()
# Producer has an internal buffer where messages are queued
# - `flush()` blocks until all messages are sent
# - Ensures messages are actually delivered to Kafka broker

1. Create Producer and connect to Kafka
2. `produce()` puts message in internal buffer (async)
3. `flush()` forces all buffered messages to be sent (sync)
4. Kafka broker receives and stores message in 'my-topic'

## Partitions

- Partitions enable **parallelism** in Kafka. 
- Each partition is **an ordered log of messages**, and messages within a partition are assigned a **sequential id** called an **offset**.

Key characteristics:  
- Partitions are distributed across brokers for *fault tolerance*
- Each partition has **one leader** and **zero or more followers**
  - One Partition Has ONE Leader
  - All **writes** go to the partition leader
  - **Reads** default to the leader,
    - but since Kafka 2.4(KIP-392), consumers can optionally **fetch from the closest replica** (including followers) by setting `client.rack` to match the broker's `broker.rack`. This reduces cross-AZ/cross-DC network costs in cloud deployments.
  - Followers replicate the leader's log
    - Followers that fall out of ISR but haven't lost data are now tracked as **Eligible Leader Replicas (ELR)** by the KRaft controller (Kafka 4.0, KIP-966), making them safe candidates for leader election without risking data loss.
  - One Topic Can Have MULTIPLE Leaders: a topic is split into multiple partitions, and each partition has its own leader
  
  ```text
  ONE Partition → REPLICATED across MULTIPLE Brokers → BUT only ONE Leader  

  E.g. Topic: "user-events" with 3 partitions, replication factor: 3

  Partition 0:
  ├─ Broker 1: 👑 LEADER
  ├─ Broker 2: Follower (copy)
  └─ Broker 3: Follower (copy)

  Partition 1:
  ├─ Broker 1: Follower (copy)
  ├─ Broker 2: 👑 LEADER
  └─ Broker 3: Follower (copy)

  Partition 2:
  ├─ Broker 1: Follower (copy)
  ├─ Broker 2: Follower (copy)
  └─ Broker 3: 👑 LEADER
  ```
  

### Custom partitioning
1. No need to set it
   - When a **key is provided**: Kafka uses **murmur2 hashing**
     on the key to determine the partition — messages with the
     same key always go to the same partition.
   - When **key is null**:
     - Kafka < 2.4: round-robin across all partitions
     - Kafka ≥ 2.4: **sticky partitioner** (KIP-480) — messages
       batch to the same partition until the batch is full, then
       switch. This significantly improves batching efficiency
       and throughput.
2. Write partitioner function（kafka-python supported）
   - Python's hash, not kafka's murmur2
   - ‼️ Cross-language inconsistency (use with caution in production)
   - If you need murmur2 consistency with Java clients, use `murmur2` from the `kafka-python` built-in utils or the `confluent-kafka` Python client which wraps librdkafka and uses the same murmur2 implementation.

In [2]:
# Producer with custom partitioning
def custom_partitioner(key, all_partitions, available):
    return hash(key) % len(all_partitions)

# Create producer with configuration
producer = KafkaProducer(
    bootstrap_servers=["localhost:9092"],
    partitioner=custom_partitioner,
)

# Send with key to control partition
producer.send('my-topic', key=b'user-123', value=b'message data')

In [18]:
from kafka.admin import KafkaAdminClient

admin = KafkaAdminClient(
    bootstrap_servers="localhost:9092",
    client_id="admin-check"
)

# 1. check all topics
topics = admin.list_topics()
print(topics)

# 2. specific topic's metadata
metadata = admin.describe_topics(["my-topic"])
print(metadata)

['my-topic']
[{'error_code': 0, 'topic': 'my-topic', 'is_internal': False, 'partitions': [{'error_code': 0, 'partition': 0, 'leader': 1, 'replicas': [1], 'isr': [1], 'offline_replicas': []}]}]


### Expand Number of Partitions
for now, there is only one partition for topic 'my-topic':  
```bash
% docker exec -it kafka bash
[appuser@a9d42eb70e1b ~]$ kafka-topics \
        --describe \
        --topic my-topic \
        --bootstrap-server localhost:9092
Topic: my-topic TopicId: SUrOWOEDSTGwecDLa9TS_w PartitionCount: 1   ReplicationFactor: 1     Configs: 
        Topic: my-topic Partition: 0    Leader: 1       Replicas: 1 Isr: 1
```


**Step 1: Expand to 3 partitions**      
- Kafka supports increasing the number of partitions using the Admin API, 
- but **partitions cannot be reduced** because existing messages and offsets cannot be safely redistributed
- If try to reduce number of partitions, `InvalidPartitionsError` will raise
- scale partitions up, never down.
- so partition quantities are usually planned at first

what happened after expanding:    
- The data of the original partition will not be re-hashed.
- The **new partition** only receives **new messages**
- Messages with the same key may be mapped to different partitions (before vs. after expansion)

Kafka CLI:  
```bash
kafka-topics \
  --alter \
  --topic my-topic \
  --partitions 3 \
  --bootstrap-server localhost:9092
```

or KafkaAdminClient:  

In [14]:
from kafka.admin import KafkaAdminClient, NewPartitions
from kafka.errors import InvalidPartitionsError

admin = KafkaAdminClient(
    bootstrap_servers="localhost:9092",
    client_id="partition-expander"
)

topic_name = "my-topic"
new_partition_count = 3

try:
    admin.create_partitions({
        topic_name: NewPartitions(total_count=new_partition_count)
    })
    print(f"Topic '{topic_name}' expanded to {new_partition_count} partitions")
except InvalidPartitionsError as e:
    print("Cannot reduce partitions or invalid request:", e)

# check
metadata = admin.describe_topics([topic_name])
partition_count = len(metadata[0]["partitions"])

print("Partition count:", partition_count)

Topic 'my-topic' expanded to 3 partitions
Partition count: 3


**Step 2: Producer (without key)**    

Kafka does not guarantee ordering across partitions

In [16]:
import time
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: v.encode()
)

for i in range(20):
    producer.send("my-topic", value=f"msg-{i}")
    print(f"sent msg-{i}")
    time.sleep(0.1)

producer.flush()

from kafka import KafkaConsumer
# Read only 20 messages then stop
message_count = 0
max_messages = 20

consumer = KafkaConsumer(
    "my-topic",
    bootstrap_servers="localhost:9092",
    auto_offset_reset="earliest",
    enable_auto_commit=False,
)

for msg in consumer:
    print(
        f"partition={msg.partition}, "
        f"offset={msg.offset}, "
        f"value={msg.value.decode()}"
    )
    message_count += 1
    if message_count >= max_messages:
        break
consumer.close()

sent msg-0
sent msg-1
sent msg-2
sent msg-3
sent msg-4
sent msg-5
sent msg-6
sent msg-7
sent msg-8
sent msg-9
sent msg-10
sent msg-11
sent msg-12
sent msg-13
sent msg-14
sent msg-15
sent msg-16
sent msg-17
sent msg-18
sent msg-19
partition=2, offset=0, value=msg-4
partition=2, offset=1, value=msg-5
partition=2, offset=2, value=msg-6
partition=2, offset=3, value=msg-9
partition=2, offset=4, value=msg-0
partition=2, offset=5, value=msg-3
partition=2, offset=6, value=msg-7
partition=2, offset=7, value=msg-14
partition=2, offset=8, value=msg-18
partition=2, offset=9, value=msg-19
partition=0, offset=0, value=Hello World
partition=0, offset=1, value=message data
partition=0, offset=2, value=msg-8
partition=0, offset=3, value=msg-1
partition=0, offset=4, value=msg-4
partition=0, offset=5, value=msg-5
partition=0, offset=6, value=msg-9
partition=0, offset=7, value=msg-10
partition=0, offset=8, value=msg-11
partition=0, offset=9, value=msg-13


**Step 3: Producer (with key)**    

- Kafka guarantees ordering only within a single partition.
- Using a key ensures all messages with the same key go to the same partition.  

what will happen:  
- All into the same partition
- offset strictly increasing
- The order is exactly correct

In [17]:
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    key_serializer=lambda k: k.encode(),
    value_serializer=lambda v: v.encode(),
)

for i in range(10):
    producer.send(
        "my-topic",
        key="user-1",   # has key now
        value=f"msg-{20 + i}"
    )
    print(f"sent msg-{i}")

producer.flush()

# consume again
# instead of set max #messages, read Until No More Messages
consumer = KafkaConsumer(
    "my-topic",
    bootstrap_servers="localhost:9092",
    auto_offset_reset="earliest",
    enable_auto_commit=False,
    consumer_timeout_ms=1000  # Stop if no message for 1 second
)

try:
    for msg in consumer:
        print(
            f"partition={msg.partition}, "
            f"offset={msg.offset}, "
            f"value={msg.value.decode()}"
        )
except StopIteration:
    print("No more messages")

consumer.close()

sent msg-0
sent msg-1
sent msg-2
sent msg-3
sent msg-4
sent msg-5
sent msg-6
sent msg-7
sent msg-8
sent msg-9
partition=0, offset=0, value=Hello World
partition=0, offset=1, value=message data
partition=0, offset=2, value=msg-8
partition=0, offset=3, value=msg-1
partition=0, offset=4, value=msg-4
partition=0, offset=5, value=msg-5
partition=0, offset=6, value=msg-9
partition=0, offset=7, value=msg-10
partition=0, offset=8, value=msg-11
partition=0, offset=9, value=msg-13
partition=0, offset=10, value=msg-16
partition=1, offset=0, value=msg-0
partition=1, offset=1, value=msg-1
partition=1, offset=2, value=msg-2
partition=1, offset=3, value=msg-3
partition=1, offset=4, value=msg-7
partition=1, offset=5, value=msg-2
partition=1, offset=6, value=msg-6
partition=1, offset=7, value=msg-8
partition=1, offset=8, value=msg-12
partition=1, offset=9, value=msg-15
partition=1, offset=10, value=msg-17
partition=2, offset=0, value=msg-4
partition=2, offset=1, value=msg-5
partition=2, offset=2, value

## Offsets

- Offsets are **unique identifiers for each record** within a partition. 
- They represent **the position of a consumer in the partition**. 
- Kafka maintains the offset for each consumer group.


## Producers

- Producers publish data to topics. 
- They choose which partition to send data to within a topic, either **round-robin** for load balancing or based on a **semantic partition key**.
- 消息经过 Serializer → Partitioner → RecordAccumulator → Sender 线程 → Broker
- Producer 使用两个线程：**主线程**（消息创建、序列化、分区）和 **Sender 线程**（网络 I/O）
- **acks 配置**：
  - `acks=0`: 不等待确认，最快但可能丢消息
  - `acks=1`: 等待 leader 确认
  - `acks=all/-1`: 等待所有 ISR 副本确认，最安全
- **Delivery Semantics**:
  - At Most Once（最多一次）
  - At Least Once（至少一次）
  - Exactly Once（精确一次，需要 idempotent producer + transactions）

### Producer Configuration

<img src='./pic/4_Apache-Kafka---lingerms-and-batchsize.png' width=500>  

more about [linger.ms and batch.size](https://www.geeksforgeeks.org/java/apache-kafka-linger-ms-and-batch-size/)

In [5]:
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'], # Kafka broker(s) to connect to for sending messages
    value_serializer=lambda v: json.dumps(v).encode('utf-8'), # Converts Python objects to JSON bytes before sending as message values.
    key_serializer=lambda k: k.encode('utf-8') if k else None, # Converts the message key to UTF-8 bytes if a key is provided
    acks='all',  # Producer waits for all replicas of a partition to acknowledge the write for maximum durability.
    retries=3, # Number of times the producer will retry sending a failed message.
    max_in_flight_requests_per_connection=5, # Max number of unacknowledged messages the producer can send on a single connection.
    compression_type='gzip', # Compresses messages using gzip to reduce network usage.
    batch_size=16384, # Maximum size (in bytes) of a batch before sending messages to Kafka.
    linger_ms=10 # Time (in ms) to wait before sending a batch, allowing more messages to accumulate for efficiency.
)

### Synchronous vs Asynchronous Sending

1. **Synchronous** (`.get()`) 
    - Blocks the code until Kafka acknowledges the message. 
    - Easier to handle failures immediately but slower if sending many messages.
2. **Asynchronous** (`.add_callback()`/`.add_errback()`) 
    - Returns immediately; 
    - sending happens in the background. 
    - Faster for high throughput but you handle success/failure via callbacks.

What you’ll notice:  
- Sync sending prints messages one by one and takes more time because it waits for Kafka acknowledgment each time.
- Async sending prints almost immediately; messages are sent in the background, and callbacks log results. Total time is much shorter.



In [9]:
import time
from threading import Event
# ----------------------------
# Synchronous send (blocking)
# ----------------------------
print("Synchronous send of 10 messages:")
start_sync = time.time()
for i in range(10):
    message = {'msg_num': i}
    try:
        future = producer.send('my-topic', message)
        record_metadata = future.get(timeout=10)  # blocks until acked
        print(f"Sent sync msg {i} to partition {record_metadata.partition}")
    except Exception as e:
        print(f"Sync send error: {e}")
end_sync = time.time()
print(f"Total sync time: {end_sync - start_sync:.4f} seconds\n")

# ----------------------------
# Asynchronous send with callback (non-blocking)
# ----------------------------
print("Asynchronous send of 10 messages:")

done_event = Event()
pending_messages = 10

def on_send_success(record_metadata):
    global pending_messages
    print(f"Async success: Topic {record_metadata.topic}, Partition {record_metadata.partition}")
    pending_messages -= 1
    if pending_messages == 0:
        done_event.set()

def on_send_error(exc):
    global pending_messages
    print(f"Async error: {exc}")
    pending_messages -= 1
    if pending_messages == 0:
        done_event.set()

start_async = time.time()
for i in range(10):
    producer.send('my-topic', {'msg_num': i}) \
        .add_callback(on_send_success) \
        .add_errback(on_send_error)

# Wait until all async messages are acknowledged
done_event.wait()
end_async = time.time()
print(f"Total async time: {end_async - start_async:.4f} seconds")

# Close producer
producer.close()

Synchronous send of 10 messages:
Sent sync msg 0 to partition 1
Sent sync msg 1 to partition 0
Sent sync msg 2 to partition 2
Sent sync msg 3 to partition 2
Sent sync msg 4 to partition 2
Sent sync msg 5 to partition 2
Sent sync msg 6 to partition 0
Sent sync msg 7 to partition 2
Sent sync msg 8 to partition 1
Sent sync msg 9 to partition 1
Total sync time: 0.1622 seconds

Asynchronous send of 10 messages:
Async success: Topic my-topic, Partition 0
Async success: Topic my-topic, Partition 0
Async success: Topic my-topic, Partition 0
Async success: Topic my-topic, Partition 1
Async success: Topic my-topic, Partition 1
Async success: Topic my-topic, Partition 1
Async success: Topic my-topic, Partition 1
Async success: Topic my-topic, Partition 1
Async success: Topic my-topic, Partition 2
Async success: Topic my-topic, Partition 2
Total async time: 0.0184 seconds


### ⭐️ Idempotent producer & Exactly-once Semantics (IMPORTANT in interviews)

**Idempotent /aɪ'dɛmpətənt/ producers** ensure exactly-once delivery semantics within a partition by automatically **handling retries without duplicates**.  
<img src='./pic/4_ip.png' width=500> 

**Exactly-once semantics** goes beyond idempotence by using **transactions** to ensure:
- Messages are written exactly once
- Consumer offsets are committed atomically with writes
- Read-process-write cycles are atomic

| Feature| What it does| Key config| Atomic multi-topic writes|Use case|
| -------| ------------| ----------| ----------| ----------| 
| **Idempotent Producer**| Guarantees that if a message is retried due to network issues, it will not be written twice.| `enable_idempotence=True`| ❌ No|Simple deduplication|
| **Exactly-once semantics (EOS)**| Guarantees that **messages are written exactly once** to Kafka even <u>across partitions and failures</u>, usually requires idempotent producer + transactional writes.| `transactional_id` + `init_transactions()`| ✅ Yes|Complex workflows that need atomic operations across multiple topics or read-process-write patterns|



#### Idempotent Producer Experiment 

**What to observe**:    
- Even if we retry sending the same message (msg_num repeats), Kafka stores it only once per key/partition offset, thanks to idempotence.

> The kafka-python-ng library doesn't support some modern Kafka features, like idempotent producer     
> So confluent-kafka is gonna used 

Idempotent producer guarantees that **retries caused by failures** do not result in duplicate records in Kafka.

**It does NOT guarantee**:
- Deduplication of semantically identical messages
- Deduplication across restarts without transactions
- Deduplication across different producer instances

So manually send the same message twice does NOT test idempotence, Kafka will (correctly) store two records.  

**Valid failure scenarios**     
| Scenario | Idempotence protects?| 
| ----------| ---------------------| 
| Network timeout| ✅| 
| Broker restart (bash docker stop kafka, wait 2–3 seconds, docker start kafka)| ✅| 
| Leader election| ✅| 
| Produce retry (`'request.timeout.ms': 1`)| ✅| 
| Same message sent twice intentionally| ❌| 


**Issue with single-broker**:  
A single-broker Kafka cluster CANNOT reliably demonstrate duplicates caused by retries.  
Kafka duplicates occur only if all of the following are true:       
	1.	Leader writes the record to the log         
	2.	Leader crashes before sending the ack      
	3.	Producer retries      
	4.	New leader does not know the previous write happened         

In a single-broker cluster:      
- ❌ Step 4 is impossible      
- There is no other replica to become leader.          

So Kafka either:     
- Acks successfully → no retry    
- Crashes before write → retry writes once        

➡️ So NO duplicate window exists   

**Goal:**      
- Non-idempotent producer can create duplicates under retry    
- Idempotent producer does not     
- Consumers use different groups so offsets don’t interfere    
- 2-broker cluster (./infra/kafka-2broker/docker-compose.yml)    

In [44]:
! docker ps | grep kafka

bdbb57f08df0   confluentinc/cp-kafka:7.5.0       "/etc/confluent/dock…"   3 minutes ago   Up 3 minutes   0.0.0.0:9093->9093/tcp, [::]:9093->9093/tcp, 0.0.0.0:19093->19093/tcp, [::]:19093->19093/tcp   kafka-2
1c3ff503fd4e   confluentinc/cp-kafka:7.5.0       "/etc/confluent/dock…"   3 minutes ago   Up 3 minutes   0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp, 0.0.0.0:19092->19092/tcp, [::]:19092->19092/tcp   kafka-1


In [40]:
# Verify Cluster Health
# Should complete in 1-2 seconds with no warnings
! docker exec kafka-1 kafka-topics \
    --list \
    --bootstrap-server localhost:9092

In [43]:
# Verify both brokers are registered in ZooKeeper
! docker exec zookeeper zookeeper-shell localhost:2181 ls /brokers/ids

Connecting to localhost:2181

WATCHER::

WatchedEvent state:SyncConnected type:None path:null
[1, 2]


In [41]:
# Delete the old topic
! docker exec kafka-1 kafka-topics \
    --delete \
    --topic idempotence-test-1 \
    --bootstrap-server localhost:9092

Error while executing topic command : Topic 'idempotence-test-1' does not exist as expected
[2026-01-08 00:04:01,406] ERROR java.lang.IllegalArgumentException: Topic 'idempotence-test-1' does not exist as expected
	at kafka.admin.TopicCommand$.kafka$admin$TopicCommand$$ensureTopicExists(TopicCommand.scala:399)
	at kafka.admin.TopicCommand$TopicService.deleteTopic(TopicCommand.scala:359)
	at kafka.admin.TopicCommand$.main(TopicCommand.scala:64)
	at kafka.admin.TopicCommand.main(TopicCommand.scala)
 (kafka.admin.TopicCommand$)


In [46]:
# create new topic (1 partition makes duplicates easier to spot.)
! docker exec kafka-1 kafka-topics \
    --create \
    --topic idempotence-test-1 \
    --bootstrap-server localhost:9092 \
    --partitions 1 \
    --replication-factor 1

[2026-01-08 00:07:18,524] WARN [AdminClient clientId=adminclient-1] Connection to node 2 (localhost/127.0.0.1:9093) could not be established. Broker may not be available. (org.apache.kafka.clients.NetworkClient)
[2026-01-08 00:07:18,573] WARN [AdminClient clientId=adminclient-1] Connection to node 2 (localhost/127.0.0.1:9093) could not be established. Broker may not be available. (org.apache.kafka.clients.NetworkClient)
[2026-01-08 00:07:18,662] WARN [AdminClient clientId=adminclient-1] Connection to node 2 (localhost/127.0.0.1:9093) could not be established. Broker may not be available. (org.apache.kafka.clients.NetworkClient)
[2026-01-08 00:07:18,908] WARN [AdminClient clientId=adminclient-1] Connection to node 2 (localhost/127.0.0.1:9093) could not be established. Broker may not be available. (org.apache.kafka.clients.NetworkClient)
[2026-01-08 00:07:19,337] WARN [AdminClient clientId=adminclient-1] Connection to node 2 (localhost/127.0.0.1:9093) could not be established. Broker may

In [29]:
# check which is the leader
! docker exec kafka-1 kafka-topics \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic idempotence-test-1

[2026-01-07 23:57:04,503] WARN [AdminClient clientId=adminclient-1] Connection to node 2 (localhost/127.0.0.1:9093) could not be established. Broker may not be available. (org.apache.kafka.clients.NetworkClient)
Topic: idempotence-test-1	TopicId: J42GeOc-TJ2VQvf5SMYRuQ	PartitionCount: 1	ReplicationFactor: 2	Configs: min.insync.replicas=1,unclean.leader.election.enable=true
	Topic: idempotence-test-1	Partition: 0	Leader: 1	Replicas: 1,2	Isr: 1,2


<font color=red> IMPORTANT</font>  
- Leader: 1 → kafka-1 is the leader
- Leader: 2 → kafka-2 is the leader

**Producer WITHOUT idempotence (expected duplicates)** 

while producting messages, Kill leader mid-stream:  
```bash
docker kill kafka-1
# wait 3-4s
docker start kafka-1
```

In [30]:
from confluent_kafka import Producer
import time

producer = Producer({
    'bootstrap.servers': 'localhost:9092,localhost:9093',
    'enable.idempotence': False,   # ❌ wihtout idempotence
    'acks': 1, # Only wait for leader (not replicas)
    # force timeout
    'request.timeout.ms': 5000,  
    'retries': 10,
    'max.in.flight.requests.per.connection': 5,  # ✅ Multiple in-flight
    'metadata.max.age.ms': 1000,  # Fast metadata refresh
})

print("Producing WITHOUT idempotence...")

for i in range(20):
    try:
        producer.produce(
            'idempotence-test-1',
            key=str(i),
            value=f'no-idem-{i}',
            callback=lambda err, msg: print(f"  ✅ Delivered: {msg.key().decode()}" if not err else f"  ❌ Failed: {err}")
        )
        print(f"📤 Produced message {i}")
        # Kill leader after message 8
        if i == 8:
            print("\n⚠️  Now kill leader with: docker kill kafka-1 (use kill, not stop!)\n")
            time.sleep(5)  # Give time to kill it
        # Slow down the producer to have time to stop kafka
        time.sleep(1)
    except BufferError:
        print(f"⚠️  Queue full, waiting...")

time.sleep(1)
producer.flush()
print("\n✅ All messages flushed")

Producing WITHOUT idempotence...
📤 Produced message 0
📤 Produced message 1
📤 Produced message 2
📤 Produced message 3
📤 Produced message 4
📤 Produced message 5
📤 Produced message 6
📤 Produced message 7
📤 Produced message 8

⚠️  Now kill leader with: docker kill kafka-1 (use kill, not stop!)



%6|1767830253.495|FAIL|rdkafka#producer-8| [thrd:localhost:9092/1]: localhost:9092/1: Disconnected: connection closed by peer: receive 0 after POLLIN (after 8730ms in state UP)
%6|1767830253.498|FAIL|rdkafka#producer-8| [thrd:localhost:9092/1]: localhost:9092/1: Disconnected: connection reset by peer (after 1ms in state APIVERSION_QUERY)


📤 Produced message 9
📤 Produced message 10


%6|1767830260.757|FAIL|rdkafka#producer-8| [thrd:localhost:9092/1]: localhost:9092/1: Disconnected: connection reset by peer (after 7003ms in state APIVERSION_QUERY, 1 identical error(s) suppressed)


📤 Produced message 11
📤 Produced message 12
📤 Produced message 13
📤 Produced message 14
📤 Produced message 15
📤 Produced message 16
📤 Produced message 17
📤 Produced message 18
📤 Produced message 19
  ✅ Delivered: 0
  ✅ Delivered: 1
  ✅ Delivered: 2
  ✅ Delivered: 3
  ✅ Delivered: 4
  ✅ Delivered: 5
  ✅ Delivered: 6
  ✅ Delivered: 7
  ✅ Delivered: 8
  ✅ Delivered: 9
  ✅ Delivered: 10
  ✅ Delivered: 11
  ✅ Delivered: 12
  ✅ Delivered: 13
  ✅ Delivered: 14
  ✅ Delivered: 15
  ✅ Delivered: 16
  ✅ Delivered: 17
  ✅ Delivered: 18
  ✅ Delivered: 19

✅ All messages flushed


In [31]:
from confluent_kafka import Consumer
import time

# Create consumer
consumer = Consumer({
    'bootstrap.servers': 'localhost:9092,localhost:9093',  # or 9092 if running inside Docker
    'group.id': f'jupyter-consumer-{int(time.time())}',
    'auto.offset.reset': 'earliest'  # equivalent to --from-beginning
})

# Subscribe to topic
consumer.subscribe(['idempotence-test-1'])

print("Consuming messages...")
messages = []

try:
    # Poll for 10 seconds or until we get 20 messages
    timeout = time.time() + 10
    while time.time() < timeout:
        msg = consumer.poll(timeout=1.0)
        
        if msg is None:
            continue
        if msg.error():
            print(f"Error: {msg.error()}")
            continue
            
        # Decode and display message
        key = msg.key().decode('utf-8') if msg.key() else None
        value = msg.value().decode('utf-8')
        
        print(f"Key: {key}, Value: {value}, Partition: {msg.partition()}, Offset: {msg.offset()}")
        messages.append({'key': key, 'value': value, 'partition': msg.partition(), 'offset': msg.offset()})
        
except KeyboardInterrupt:
    pass
finally:
    consumer.close()

print(f"\nTotal messages consumed: {len(messages)}")

Consuming messages...


%3|1767830276.598|FAIL|rdkafka#consumer-9| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv4#127.0.0.1:9092 failed: Connection refused (after 0ms in state CONNECT)



Total messages consumed: 0


%6|1767830319.926|FAIL|rdkafka#producer-8| [thrd:localhost:9093/2]: localhost:9093/2: Disconnected: connection closed by peer: receive 0 after POLLIN (after 66421ms in state UP)
%3|1767830320.060|FAIL|rdkafka#producer-8| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv6#[::1]:9092 failed: Connection refused (after 1ms in state CONNECT)
%3|1767830321.060|FAIL|rdkafka#producer-8| [thrd:localhost:9093/2]: localhost:9093/2: Connect to ipv6#[::1]:9093 failed: Connection refused (after 0ms in state CONNECT)
%3|1767830321.117|FAIL|rdkafka#producer-8| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv6#[::1]:9092 failed: Connection refused (after 1ms in state CONNECT, 1 identical error(s) suppressed)
%3|1767830322.072|FAIL|rdkafka#producer-8| [thrd:localhost:9093/2]: localhost:9093/2: Connect to ipv6#[::1]:9093 failed: Connection refused (after 9ms in state CONNECT, 1 identical error(s) suppressed)
%3|1767830322.120|FAIL|rdkafka#producer-8| [thrd

## Consumers

- Consumers read data from topics. 
- They subscribe to **one or more topics** and process the feed of published records.

### Consumer Configuration


In [35]:
from kafka import KafkaConsumer
consumer = KafkaConsumer(
    'my-topic',
    bootstrap_servers=['localhost:9092'],
    group_id='my-consumer-group-1', # Kafka tracks consumer offsets per consumer group
    auto_offset_reset='earliest',   # or latest
    enable_auto_commit=True,
    auto_commit_interval_ms=5000,
    max_poll_records=500,
    
    # Reduce rebalancing time
    session_timeout_ms=10000,      # Reduced from 30s to 10s
    heartbeat_interval_ms=3000,    # Keep at 1/3 of session timeout
    
    # Optimize fetch performance
    fetch_min_bytes=1024,          # Wait for at least 1KB
    fetch_max_wait_ms=100,         # Don't wait more than 100ms
    max_partition_fetch_bytes=1048576,  # 1MB per partition
    
    # this deserializer expects all messages to be JSON
    # may hit JSONDecodeError when message is Empty/None, Not valid JSON format, or Binary data that isn't UTF-8 text
    # value_deserializer=lambda m: json.loads(m.decode('utf-8'))
)

consumer

In [ ]:
def reset_consumer(topic = 'my-topic', group_id='my-group'):
    consumer = KafkaConsumer(
        topic,
        bootstrap_servers=['localhost:9092'],
        group_id=group_id,
        auto_offset_reset='earliest',
        enable_auto_commit=False  # Don't save offset
    )

    # Force seek to beginning
    consumer.poll(0)  # Trigger partition assignment
    consumer.seek_to_beginning()

    print("Consumer reset to beginning")
    return consumer

### Consuming Messages


In [11]:
consumer = reset_consumer(topic='my-topic')
print("Consumer starting, waiting for partition assignment...")
consumer.poll(timeout_ms=0)  # Trigger assignment
print(f"Assigned partitions: {consumer.assignment()}")

# Simple consumption loop
# Infinite iterator, won't stop untill manually interrupt it
for message in consumer:    
    print(f"Topic: {message.topic}")
    print(f"Partition: {message.partition}")
    print(f"Offset: {message.offset}")
    print(f"Key: {message.key}")
    print(f"Value: {message.value}")
    print(f"Timestamp: {message.timestamp}")

Consumer starting, waiting for partition assignment...
Assigned partitions: {TopicPartition(topic='my-topic', partition=0)}
Topic: my-topic
Partition: 0
Offset: 0
Key: None
Value: b'Hello World'
Timestamp: 1768065215390
Topic: my-topic
Partition: 0
Offset: 1
Key: b'user-123'
Value: b'message data'
Timestamp: 1768065223101


KeyboardInterrupt: 

This simple consumption loop will NOT stop automatically. It runs indefinitely until you manually interrupt it. It:   
- Continuously polls Kafka for new messages
- Waits indefinitely if no messages are available
- Only stops when you press Ctrl+C or kill the process

To stop it:

In [43]:
# Solution 1: Process N Messages Then Stop
consumer = reset_consumer(topic='my-topic')

max_messages = 10
count = 0

for message in consumer:
    print(f"Value: {message.value}")
    count += 1
    if count >= max_messages:
        break

print(f"Processed {count} messages")
consumer.close()

Value: b'Hello World'
Value: b'message data'
Value: b'msg-8'
Value: b'msg-1'
Value: b'msg-4'
Value: b'msg-5'
Value: b'msg-9'
Value: b'msg-10'
Value: b'msg-11'
Value: b'msg-13'
Processed 10 messages


In [45]:
# Solution 2: poll-based consumption, Use poll() with Timeout
consumer = reset_consumer(topic='my-topic')

import time
timeout_seconds = 10
start_time = time.time()

def process_message(message):
    key = message.key.decode("utf-8") if message.key else None
    value = message.value.decode("utf-8") if message.value else None

    print(
        f"[{message.topic} | p{message.partition} | offset {message.offset}] "
        f"key={key}, value={value}, ts={message.timestamp}"
    )
while True:
    msg_pack = consumer.poll(timeout_ms=1000) # Wait 1 second for messages
    if not msg_pack:
        print("No messages received")
    for topic_partition, messages in msg_pack.items():
        for message in messages:
            process_message(message)
    # Stop after timeout
    if time.time() - start_time > timeout_seconds:
        print("Timeout reached")
        break

consumer.close()

[my-topic | p1 | offset 0] key=None, value=msg-0, ts=1768263301634
[my-topic | p1 | offset 1] key=None, value=msg-1, ts=1768263301739
[my-topic | p1 | offset 2] key=None, value=msg-2, ts=1768263301845
[my-topic | p1 | offset 3] key=None, value=msg-3, ts=1768263301950
[my-topic | p1 | offset 4] key=None, value=msg-7, ts=1768263302364
[my-topic | p1 | offset 5] key=None, value=msg-2, ts=1768263349404
[my-topic | p1 | offset 6] key=None, value=msg-6, ts=1768263349816
[my-topic | p1 | offset 7] key=None, value=msg-8, ts=1768263350027
[my-topic | p1 | offset 8] key=None, value=msg-12, ts=1768263350444
[my-topic | p1 | offset 9] key=None, value=msg-15, ts=1768263350747
[my-topic | p1 | offset 10] key=None, value=msg-17, ts=1768263350953
[my-topic | p0 | offset 0] key=None, value=Hello World, ts=1768065215390
[my-topic | p0 | offset 1] key=user-123, value=message data, ts=1768065223101
[my-topic | p0 | offset 2] key=None, value=msg-8, ts=1768263302469
[my-topic | p0 | offset 3] key=None, valu

`consumer.poll()` with a timeout: 
- doesn't guarantee it will keep trying for the full duration. 
- It's a single poll attempt that waits up to 1 second, not a cumulative timeout.

what's happening
- If no messages arrive in that 1 second, it returns empty {}
- Then the outer while loop checks the 10-second timeout
- But poll() already returned, so it checks again immediately

In [47]:
# Solution 3: Consume Until Caught Up (No New Messages)
from kafka import TopicPartition

# Create consumer WITHOUT subscribing to topics
consumer = KafkaConsumer(
    # 'my-topic', # automatically subscribes to that topic
    # will conflict with manual partition assignment `consumer.assign(partitions)`
    # so no need here
    bootstrap_servers=['localhost:9092'],
    auto_offset_reset='earliest',  # Start from beginning if no committed offset
    enable_auto_commit=False  # Manual control for this example
)

# Get end offsets for all partitions for 'my-topic' 
partitions = [TopicPartition('my-topic', p) 
              for p in consumer.partitions_for_topic('my-topic')]
# Manually assign partitions (instead of subscribing)
consumer.assign(partitions)
# Get end offsets for all partitions
end_offsets = consumer.end_offsets(partitions)
print(f"End offsets: {end_offsets}")

for message in consumer:
    print(f"Offset: {message.offset}, Value: {message.value}")
    
    # Check if we've reached the end of all partitions
    tp = TopicPartition(message.topic, message.partition)
    if message.offset + 1 >= end_offsets[tp]:
        # Check if all partitions are caught up
        current_positions = {p: consumer.position(p) for p in partitions}
        if all(current_positions[p] >= end_offsets[p] for p in partitions):
            print("Caught up to end of all partitions")
            break

consumer.close()

End offsets: {TopicPartition(topic='my-topic', partition=0): 11, TopicPartition(topic='my-topic', partition=1): 11, TopicPartition(topic='my-topic', partition=2): 20}
Offset: 0, Value: b'msg-4'
Offset: 1, Value: b'msg-5'
Offset: 2, Value: b'msg-6'
Offset: 3, Value: b'msg-9'
Offset: 4, Value: b'msg-0'
Offset: 5, Value: b'msg-3'
Offset: 6, Value: b'msg-7'
Offset: 7, Value: b'msg-14'
Offset: 8, Value: b'msg-18'
Offset: 9, Value: b'msg-19'
Offset: 10, Value: b'msg-20'
Offset: 11, Value: b'msg-21'
Offset: 12, Value: b'msg-22'
Offset: 13, Value: b'msg-23'
Offset: 14, Value: b'msg-24'
Offset: 15, Value: b'msg-25'
Offset: 16, Value: b'msg-26'
Offset: 17, Value: b'msg-27'
Offset: 18, Value: b'msg-28'
Offset: 19, Value: b'msg-29'
Offset: 0, Value: b'Hello World'
Offset: 1, Value: b'message data'
Offset: 2, Value: b'msg-8'
Offset: 3, Value: b'msg-1'
Offset: 4, Value: b'msg-4'
Offset: 5, Value: b'msg-5'
Offset: 6, Value: b'msg-9'
Offset: 7, Value: b'msg-10'
Offset: 8, Value: b'msg-11'
Offset: 9, V

In [52]:
# Solution 4.1: Time Limit without Threading
consumer = reset_consumer(topic='my-topic')

import time

duration_seconds = 30
start_time = time.time()
message_count = 0

for message in consumer:
    print(f"Value: {message.value}")
    message_count += 1
    
    # Check time limit every message
    if time.time() - start_time > duration_seconds:
        print(f"Time limit reached: {duration_seconds}s")
        break

consumer.close()
print(f"Processed {message_count} messages in {time.time() - start_time:.2f}s")

Consumer reset to beginning
Value: b'msg-0'
Value: b'msg-1'
Value: b'msg-2'
Value: b'msg-3'
Value: b'msg-7'
Value: b'msg-2'
Value: b'msg-6'
Value: b'msg-8'
Value: b'msg-12'
Value: b'msg-15'
Value: b'msg-17'
Value: b'Hello World'
Value: b'message data'
Value: b'msg-8'
Value: b'msg-1'
Value: b'msg-4'
Value: b'msg-5'
Value: b'msg-9'
Value: b'msg-10'
Value: b'msg-11'
Value: b'msg-13'
Value: b'msg-16'
Value: b'msg-4'
Value: b'msg-5'
Value: b'msg-6'
Value: b'msg-9'
Value: b'msg-0'
Value: b'msg-3'
Value: b'msg-7'
Value: b'msg-14'
Value: b'msg-18'
Value: b'msg-19'
Value: b'msg-20'
Value: b'msg-21'
Value: b'msg-22'
Value: b'msg-23'
Value: b'msg-24'
Value: b'msg-25'
Value: b'msg-26'
Value: b'msg-27'
Value: b'msg-28'
Value: b'msg-29'


KeyboardInterrupt: 

`for message in consumer:` is an infinite iterator that keeps polling Kafka internally. The time check only happens after a message arrives, so if messages keep flowing, it processes them before checking the time.  So use `consumer.poll()` or with threading:

In [53]:
# Solution 4: Time Limit with Threading
consumer = reset_consumer(topic='my-topic')

import time
import threading

duration_seconds = 30
message_count = 0
stop_flag = threading.Event()

# Timer thread to set stop flag
def timeout_handler():
    time.sleep(duration_seconds)
    print(f"\nTime limit reached: {duration_seconds}s")
    stop_flag.set()

timer = threading.Thread(target=timeout_handler, daemon=True)
timer.start()

# Consume until stop flag is set
for message in consumer:
    if stop_flag.is_set():
        break
    
    print(f"Value: {message.value}")
    message_count += 1

consumer.close()
print(f"Processed {message_count} messages")

Consumer reset to beginning
Value: b'Hello World'
Value: b'message data'
Value: b'msg-8'
Value: b'msg-1'
Value: b'msg-4'
Value: b'msg-5'
Value: b'msg-9'
Value: b'msg-10'
Value: b'msg-11'
Value: b'msg-13'
Value: b'msg-16'
Value: b'msg-4'
Value: b'msg-5'
Value: b'msg-6'
Value: b'msg-9'
Value: b'msg-0'
Value: b'msg-3'
Value: b'msg-7'
Value: b'msg-14'
Value: b'msg-18'
Value: b'msg-19'
Value: b'msg-20'
Value: b'msg-21'
Value: b'msg-22'
Value: b'msg-23'
Value: b'msg-24'
Value: b'msg-25'
Value: b'msg-26'
Value: b'msg-27'
Value: b'msg-28'
Value: b'msg-29'
Value: b'msg-0'
Value: b'msg-1'
Value: b'msg-2'
Value: b'msg-3'
Value: b'msg-7'
Value: b'msg-2'
Value: b'msg-6'
Value: b'msg-8'
Value: b'msg-12'
Value: b'msg-15'
Value: b'msg-17'

Time limit reached: 30s


KeyboardInterrupt: 

still waiting for the next message, so use `consumer.poll()`!!!   

| Approach| Best For| Avoid For| 
| --------| --------| ---------| 
| `poll()`| ✅ Production systems ✅ Batch processing ✅ Graceful shutdown ✅ Monitoring/metrics| Quick scripts that don't need control| 
| `for message in consumer:`| ✅ Simple scripts ✅ Quick tests ✅ Tutorials| ❌ Production❌ Services that need shutdown❌ Long-running processes| 

### Offset Management  

In Kafka, an offset is a **unique sequential ID number assigned to each message within a partition**. It's like a bookmark that tracks **the position of messages**.  

Offset management is the process of **tracking and controlling which messages** a consumer has read and which ones it still needs to process.  

Think of it like reading a book - the offset is your page number, and offset management is how you keep track of where you left off *so you can continue reading* from the right place later.  

**Why Do We Need Offset Management?**    
1. Ensuring **Exactly-Once or At-Least-Once** Processing     
    - Without proper offset management, you could:

      - Miss messages (some data never gets processed)
      - Process duplicates (same message processed multiple times)
      - Lose progress when a consumer crashes    

    - Proper offset management ensures you **can control your delivery semantics** based on your application's needs.
2. **Consumer Recovery and Fault Tolerance**
    - When a consumer crashes or restarts, offset management allows it to:  
      - Resume from where it left off (avoiding reprocessing)
      - Reprocess messages from a specific point if needed
      - Handle failures gracefully without data loss

3. **Parallel Processing and Consumer Groups**
    - In consumer groups where multiple consumers read from different partitions:   
      - **Each consumer** tracks its **own offset per partition**
      - Offsets prevent different consumers from reading the same messages
      - Enables scalable, distributed message processing

4. **Flexibility** in Message Consumption
    - Offset management gives you control to:
      - **Replay messages** by resetting offsets to an earlier position
      - **Skip problematic messages** by advancing the offset
      - **Start from** beginning, end, or specific timestamp
      - **Audit and monitor** consumer progress

**How Kafka Manages Offsets**      
Kafka provides several ways to manage offsets:
- **Auto-commit (default)**
  - Kafka automatically commits offsets at regular intervals. 
  - Simple but may lead to message loss or duplicates if consumer crashes between commits.
- **Manual commit**
  - You explicitly commit offsets after processing messages, 
  - giving you fine-grained control over when processing is considered complete.
- **Offset storage**
  - Kafka stores committed offsets in a special internal topic called `__consumer_offsets`, which is replicated for durability.


**Real-World Example**     
Imagine an e-commerce order processing system:

- Orders come into Kafka as messages
- A consumer processes each order (payment, inventory, shipping)
- After successfully processing order #100, you commit offset 100
- If the consumer crashes, it restarts from offset 101
- Without offset management, you might charge customers twice or miss orders entirely

Proper offset management is essential for building reliable, fault-tolerant data pipelines with Kafka.

In [62]:
# Manual offset control - Batch Commit Version
from kafka import KafkaConsumer, TopicPartition, OffsetAndMetadata
import time

def process_message(message):
    key = message.key.decode("utf-8") if message.key else None
    value = message.value.decode("utf-8") if message.value else None
    
    print(
        f"[{message.topic} | p{message.partition} | offset {message.offset}] "
        f"key={key}, value={value}"
    )

consumer = KafkaConsumer(
    bootstrap_servers=['localhost:9092'],
    enable_auto_commit=False,
    group_id='my-consumer-group-1'  # required
    # if you want to commit offsets, 
    # you MUST have a group_id because Kafka stores committed offsets per consumer group.
)

# Seek to specific offset
partition = TopicPartition('my-topic', 0) # topic, partition number
consumer.assign([partition])
consumer.seek(partition, 3) # offset

# Control parameters
duration_seconds = 30
start_time = time.time()
message_count = 0
max_messages = 10

print(f"Consumer started from offset 3")

try:
    while time.time() - start_time < duration_seconds and message_count < max_messages:
        msg_pack = consumer.poll(timeout_ms=1000)
        
        if msg_pack:
            batch_offsets = {}
            batch_processed = 0
            
            for topic_partition, messages in msg_pack.items():
                for message in messages:
                    # Check limits before processing
                    if message_count >= max_messages:
                        break
                    if time.time() - start_time >= duration_seconds:
                        break
                    
                    try:
                        process_message(message)
                        message_count += 1
                        batch_processed += 1
                        
                        # Track offset for batch commit
                        batch_offsets[topic_partition] = OffsetAndMetadata(
                            message.offset + 1, None
                        )
                        
                    except Exception as e:
                        print(f"Error processing message: {e}")
                        # Don't commit this batch on error
                        consumer.seek_to_committed()
                        batch_offsets = {}
                        break
            
            # Commit entire batch at once (more efficient)
            if batch_offsets:
                consumer.commit(offsets=batch_offsets)
                print(f"Committed batch of {batch_processed} messages")

finally:
    consumer.close()
    elapsed = time.time() - start_time
    print(f"Consumer stopped. Processed {message_count} messages in {elapsed:.2f}s")

Consumer started from offset 3
[my-topic | p0 | offset 3] key=None, value=msg-1
[my-topic | p0 | offset 4] key=None, value=msg-4
[my-topic | p0 | offset 5] key=None, value=msg-5
[my-topic | p0 | offset 6] key=None, value=msg-9
[my-topic | p0 | offset 7] key=None, value=msg-10
[my-topic | p0 | offset 8] key=None, value=msg-11
[my-topic | p0 | offset 9] key=None, value=msg-13
[my-topic | p0 | offset 10] key=None, value=msg-16
Committed batch of 8 messages
Consumer stopped. Processed 8 messages in 30.10s


### Consumer Group Rebalancing

When consumers join or leave a group, Kafka rebalances partitions among the remaining consumers.

Consumers are grouped together for any one topic, and the partitions within the topic are assigned across these consumers. Consumer group rebalance can be triggered by a number of factors as the **participants of the group change, which leads to the reassignment of partitions across the consumers**.   

During a rebalance **message processing is paused**, impacting throughput.  

Group membership is managed on the broker side, and partition assignment is managed on the client side. The broker has no knowledge of what the resources are and how they are assigned amongst the consumers.


In [ ]:
from kafka import KafkaConsumer
from kafka.structs import TopicPartition

def on_assign(consumer, partitions):
    print(f"Partitions assigned: {partitions}")

def on_revoke(consumer, partitions):
    print(f"Partitions revoked: {partitions}")
    # Save state before rebalance

consumer = KafkaConsumer(
    'my-topic',
    bootstrap_servers=['localhost:9092'],
    group_id='my-group'
)

consumer.subscribe(['my-topic'], on_assign=on_assign, on_revoke=on_revoke)

## Consumer Groups

- Consumer groups allow multiple consumers to divide the work of consuming and processing records. 
- Each consumer in a group processes records from **a unique subset of partitions**.

```python
# Multiple consumers in same group
consumer1 = KafkaConsumer(
    'my-topic',
    bootstrap_servers=['localhost:9092'],
    group_id='shared-group',
    auto_offset_reset='earliest'
)

consumer2 = KafkaConsumer(
    'my-topic',
    bootstrap_servers=['localhost:9092'],
    group_id='shared-group',
    auto_offset_reset='earliest'
)
```



## Workflow step by step
<img src='./pic/4_kafka_architecture_partitions.png' width=700>

```text
1. Producer connects to any broker (broker1)
   ├─> Broker1: "I want to send to 'user-events'"
   
2. Broker1 checks metadata
   ├─> "user-events has 3 partitions"
   ├─> "Based on key 'user-123', this goes to Partition 1"
   ├─> "Partition 1 leader is Broker 2"
   
3. Broker1 redirects producer to Broker 2
   └─> "Send to Broker 2, it's the leader for Partition 1"
   
4. Producer sends to Broker 2 (leader)
   
5. Broker 2 (leader) receives message
   ├─> Assigns offset: 1000
   ├─> Writes to disk: /var/kafka-logs/user-events-1/00000.log
   ├─> Sends to Follower Brokers (3 and 1)
   
6. Broker 3 (follower) receives and writes to disk
   
7. Broker 1 (follower) receives and writes to disk
   
8. All replicas confirm to Broker 2
   
9. Broker 2 sends ACK back to producer
   └─> "Message written successfully at offset 1000"
```

## Replication and Fault Tolerance

### Replication Factor

- Each partition has a configurable replication factor. 
- One replica is the leader, and others are followers.

```bash
# Create topic with replication factor 3
kafka-topics.sh --create \
  --bootstrap-server localhost:9092 \
  --topic replicated-topic \
  --partitions 3 \
  --replication-factor 3
```



### In-Sync Replicas (ISR)

- ISR is the set of replicas that **are fully caught up with the leader**. 
- Only ISR members **are eligible to become leaders**.

```python
# Producer with acks='all' waits for all ISR to acknowledge
producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    acks='all',  # Wait for all in-sync replicas
    min_insync_replicas=2  # Minimum ISR required
)
```



## Kafka Delivery Semantics

<img src='./pic/4_exactly-once-semantics-with-apache-kafka_1.webp' width=600>

Kafka provides three types of delivery guarantees:

### 1. At-Most-Once Semantics   
- Messages may be lost but are never redelivered.         
- How it works:
  - Producer sends message without waiting for acknowledgment (acks=0)
  - Consumer commits offset before processing the message
  - If failure occurs during processing, message is lost
- Use case: When losing some data is acceptable (e.g., metrics, logs where occasional loss is tolerable)
  ```python
  # Producer configuration
  producer = KafkaProducer(
      acks=0,  # Don't wait for acknowledgment
      retries=0
  )

  # Consumer pattern
  for message in consumer:
      consumer.commit()  # Commit first
      process(message)   # Process after - risk of loss if this fails
  ```

### 2. At-Least-Once Semantics (Default)
- Messages are never lost but may be redelivered.
- How it works:
  - Producer waits for acknowledgment and retries on failure (acks=all)
  - Consumer processes message before committing offset
  - If consumer fails after processing but before commit, message is reprocessed
- Use case: Most common choice when you can handle duplicates
  ```python
  # Producer configuration
  producer = KafkaProducer(
      acks='all',  # Wait for all replicas
      retries=3,
      enable_idempotence=False
  )

  # Consumer pattern
  for message in consumer:
      process(message)      # Process first
      consumer.commit()     # Commit after - may reprocess on failure
  ```

### 3. Exactly-Once Semantics (EOS)
- Messages are delivered exactly once - no loss, no duplicates.
- How it works:
  - Uses idempotent producer + transactions
  - Atomic writes across multiple partitions
  - Transactional reads in consumer
- Use case: Financial transactions, billing, critical data pipelines
  ```python
  # Producer configuration
  producer = KafkaProducer(
      acks='all',
      enable_idempotence=True,
      transactional_id='my-transactional-id',
      max_in_flight_requests_per_connection=5
  )

  # Initialize transactions
  producer.init_transactions()

  try:
      producer.begin_transaction()
      producer.send('topic', b'message')
      producer.commit_transaction()
  except Exception:
      producer.abort_transaction()

  # Consumer configuration
  consumer = KafkaConsumer(
      isolation_level='read_committed',  # Only read committed messages
      enable_auto_commit=False
  )
  ```
<img src='./pic/4_kafka_EOS.webp' width=600>

### Key Configuration Parameters
Producer:

- `acks`: 0 (no ack), 1 (leader only), all/-1 (all replicas)
- `enable_idempotence`: Ensures producer doesn't create duplicates
- `transactional_id`: Enables exactly-once semantics
- `retries`: Number of retry attempts

Consumer:

- `enable_auto_commit`: Auto vs manual offset commit
- `isolation_level`: read_uncommitted vs read_committed
- `auto_offset_reset`: earliest, latest, none

## Performance Optimization

### Batching

Batching improves throughput by sending multiple records together.

```python
producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    batch_size=32768,  # 32KB batch size
    linger_ms=20  # Wait up to 20ms to fill batch
)
```

### Compression

Compression reduces network bandwidth and storage requirements.

```python
producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    compression_type='gzip'  # or 'snappy', 'lz4', 'zstd'
)
```

### Consumer Performance

```python
consumer = KafkaConsumer(
    'my-topic',
    bootstrap_servers=['localhost:9092'],
    max_poll_records=1000,  # Fetch more records per poll
    fetch_min_bytes=1024,   # Wait for at least 1KB
    fetch_max_wait_ms=500   # Max wait time for fetch_min_bytes
)
```



## Kafka Streams

Kafka Streams is a client library for **building stream processing** applications that transform and enrich data.


In [ ]:
# Using kafka-python for basic streaming
from kafka import KafkaConsumer, KafkaProducer
import json

consumer = KafkaConsumer(
    'input-topic',
    bootstrap_servers=['localhost:9092'],
    value_deserializer=lambda m: json.loads(m.decode('utf-8'))
)

producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

# Stream processing: filter and transform
for message in consumer:
    data = message.value
    if data['temperature'] > 30:
        transformed = {
            'sensor_id': data['sensor_id'],
            'alert': 'HIGH_TEMPERATURE',
            'value': data['temperature']
        }
        producer.send('alerts-topic', transformed)

In [ ]:
from __future__ import annotations
import time
from dataclasses import dataclass, field
from enum import Enum
from threading import Lock
from typing import Any, Callable, Dict, List, Optional, Set, Tuple
from concurrent.futures import FIRST_COMPLETED, Future, ThreadPoolExecutor, wait

class TaskState(str, Enum):
    PENDING = 'PENDING'
    RUNNING = 'RUNNING'
    SUCCESS = 'SUCCESS'
    FAILED = 'FAILED'
    RETRYING = 'RETRYING'
    SKIPPED = 'SKIPPED'

@dataclass(frozen=True)
class Task:
    task_id: str
    func: Callable[..., Any]
    arg: Tuple[Any, ...] = field(default_factory=tuple)
    kwargs: Dict[str, Any] = field(default_factory=dict)
    dependencies: List[str] = field(default_factory=list)
    retries: int = 0
    retry_delay: int = 0

    @staticmethod
    def from_dict(data: Dict[str, Any]) -> 'Task':
        task_id = data['task_id']
        func = data['func']

        if not callable(func):
            raise TypeError(f"Task '{task_id}' func must be callable")
        retries = data.get('retries', 0)
        retry_delay = data.get('retry_delay', 0)
        if retries<0:
            raise ValueError(f"Task '{task_id}' retries cannot be negative")
        dependencies = list(dict.fromkeys(data.get('dependencies', [])))

        return Task(
            task_id=task_id,
            func=func,
            args = tuple(data.get('args', ())),
            kwargs=dict(data.get('kwargs', {})),
            dependencies=dependencies,
            retries=retries,
            retry_delay=retry_delay,
        )

@dataclass
class TaskRuntime:
    state: TaskState = TaskState.PENDING
    attempts: int = 0
    last_error: Optional[BaseException] = None

@dataclass
class TaskExecutionResult:
    task_id: str
    state: TaskState
    attempts: int
    error: Optional[BaseException] = None

class TaskScheduler:
    def __init__(self, max_workers: int = 4) -> None:
        if max_workers <= 0:
            raise ValueError('max_workers must be >= 1')
        self.max_workers = max_workers
        self.tasks: Dict[str, Task] = {}
        self.runtime: Dict[str, Set[str]] = {}
        self._lock = Lock()
    
    def add_task(self, task_data: Dict[str, Any]) -> None:
        task = Task.from_dict(task_data)

        with self._lock:
            if task.task_id in self.tasks:
                raise ValueError(f"Duplicate task_id detected: '{task.task_id}'")
            self.tasks[task.task_id] = task
            self.runtime[task.task_id] = TaskRuntime
    
    def run(self)->Dict[str, TaskRuntime]:
        self._validate_graph()
        self._build_dependents_map()

        if not self.tasks:
            return self.runtime
        
        in_flight: Dict[Future, str] = {}
        submitted: Set[str] = set()

        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            while True:
                self._mark_blocked_tasks_as_skipped()
                ready_tasks = self.__get_ready_tasks(submitted)
                for task_id in ready_tasks:
                    with self._lock:
                        self.runtime[task_id].state = TaskState.RUNNING
                    future = executor.submit(self._execute_task, self.tasks[task_id])
                    in_flight[future] = task_id
                    submitted.add(task_id)
                if not in_flight:
                    if self._all_terminal():
                        break
                    else:
                        raise RuntimeError("Scheduler stuck")
                
                done, _ = wait(in_flight.keys(), return_when=FIRST_COMPLETED)
                for future in done:
                    task_id = in_flight.pop(future)
                    result = future.result()

                    with self._lock:
                        runtime = self.runtime[task_id]
                        runtime.state = result.state
                        runtime.attempts = result.attempts
                        runtime.last_error = result.error
        
        return self.runtime
    
    def get_task_states(self) -> Dict[str, TaskState]:
        with self._lock:
            return {task_id: rt.state for task_id, rt in self.runtime.items()}
    
    def _validate_graph(self)-> None:
        with self._lock:
            tasks_snapshot = dict(self.tasks)
        
        for task_id, task in tasks_snapshot.items():
            for dep in task.dependencies:
                if dep not in tasks_snapshot:
                    raise ValueError(f"Task '{task_id}' depends on missing task '{dep}'")
        self._detect_cycle(tasks_snapshot)
    
    def _detect_cycle(self, tasks_snapshot:Dict[str, Task])->None:
        visited: Set[str] = set()
        visiting: Set[str] = set()

        def dfs(task_id: str) -> None:
            if task_id in visiting:
                raise ValueError(f"Cycle detected involving task '{task_id}'")
            if task_id in visited:
                return
            
            visiting.add(task_id)
            for dep in tasks_snapshot[task_id].dependencies:
                dfs(dep)
            visiting.remove(task_id)
            visited.add(task_id)
        
        for task_id in tasks_snapshot:
            if task_id not in visited:
                dfs(task_id)
    
    def _build_dependents_map(self) -> None:
        

## Best Practices

### Producer Best Practices

1. Use appropriate acks setting based on durability requirements
2. Enable idempotence for exactly-once semantics
3. Configure retries and timeouts properly
4. Use compression for large messages
5. Implement proper error handling and callbacks

```python
producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    acks='all',
    enable_idempotence=True,
    retries=3,
    max_in_flight_requests_per_connection=5,
    compression_type='snappy',
    request_timeout_ms=30000
)

def send_with_retry(topic, key, value, max_attempts=3):
    for attempt in range(max_attempts):
        try:
            future = producer.send(topic, key=key, value=value)
            record_metadata = future.get(timeout=10)
            return record_metadata
        except Exception as e:
            if attempt == max_attempts - 1:
                raise
            time.sleep(2 ** attempt)
```

### Consumer Best Practices

1. Use consumer groups for scalability
2. Handle rebalancing gracefully
3. Commit offsets appropriately
4. Implement proper error handling
5. Monitor consumer lag

```python
from kafka import KafkaConsumer
from kafka.errors import KafkaError

consumer = KafkaConsumer(
    'my-topic',
    bootstrap_servers=['localhost:9092'],
    group_id='my-group',
    enable_auto_commit=False,
    max_poll_interval_ms=300000
)

def process_messages():
    try:
        for message in consumer:
            try:
                process_message(message)
                consumer.commit()
            except Exception as e:
                log_error(e, message)
                # Implement dead letter queue
                send_to_dlq(message)
    except KafkaError as e:
        log_error(e)
    finally:
        consumer.close()
```



## Advanced Patterns

### Exactly-Once Semantics

```python
from kafka import KafkaProducer, KafkaConsumer

# Producer with transactions
producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    transactional_id='my-transactional-id',
    enable_idempotence=True
)

producer.init_transactions()

try:
    producer.begin_transaction()
    producer.send('output-topic', b'message1')
    producer.send('output-topic', b'message2')
    producer.commit_transaction()
except Exception as e:
    producer.abort_transaction()
```

### Dead Letter Queue Pattern

```python
def process_with_dlq(consumer, dlq_producer):
    for message in consumer:
        try:
            process_message(message)
            consumer.commit()
        except Exception as e:
            # Send to dead letter queue
            dlq_message = {
                'original_topic': message.topic,
                'original_partition': message.partition,
                'original_offset': message.offset,
                'error': str(e),
                'data': message.value
            }
            dlq_producer.send('dlq-topic', dlq_message)
            consumer.commit()
```



# Lag Spike
A lag spike happens when producers generate messages faster than consumers can process them for a short period of time.  

Why lag spikes happen (intuition)   

| Cause | What it means| 
| -------| --------------| 
| Traffic burst| Producers outpace consumers| 
| Slow consumer| CPU, I/O, or external API| 
| Consumer pause| GC, rebalance, outage| 
| Partition bottleneck| Too few partitions| 


# Install Kafka
## Kafka Server (Docker Compose)

### method 1 (+ Zookeeper)
1. create docker-compose.yml
   ```bash
    # 1. create new dictory
    mkdir infra/kafka-docker
    cd infra/kafka-docker

    # 2. create docker-compose.yml
    touch docker-compose.yml

    # 3. open the file
    code docker-compose.yml     # VS Code
    # or
    vim docker-compose.yml
   ```
2. start kafka
   ```bash
   docker compose up -d
   # check 
   docker ps
   # should see zookeeper and kafka started
   ```
3. check if kafka is available
   ```bash
   # enter kafka container
   docker exec -it kafka bash
   
   # create topic
   kafka-topics --create \
      --topic my-topic \
      --bootstrap-server localhost:9092 \
      --partitions 1 \
      --replication-factor 1
   
   # produce topic
   kafka-console-producer \
      --topic my-topic \
      --bootstrap-server localhost:9092
   # input:
   hello
   world

   # consume topic (in another terminal)
   kafka-console-consumer \
      --topic my-topic \
      --from-beginning \
      --bootstrap-server localhost:9092
   # if see hello world, kafka runs successfully
   ```
4. to exit kafka container (but not stop kafka)
   ```bash
   [appuser@7bfc7b21a403 ~]$ exit
   ```
5. to stop kafka in docker
   ```bash
   # stop only one container
   docker stop kafka
   # stop all containers in that compose
   # in dir of docker-compose.yml
   docker compose stop
   # delete compose
   docker compose down
   ```



### method 2 (Kafka KRaft, no Zookeeper)
- for Kafka ≥ 3.x
- kafka's future default architecture.  
- KRaft replaces ZooKeeper by embedding metadata management into Kafka itself, simplifying architecture and improving scalability and operability.

1. create docker-compose.yml
   ```bash
    # 1. create new dictory
    mkdir infra/kafka-docker
    cd infra/kafka-docker

    # 2. create docker-compose.yml
    touch docker-compose.yml

    # 3. open the file
    code docker-compose.yml     # VS Code
    # or
    vim docker-compose.yml
   ```
2. start kafka (kraft)
   ```bash
   docker compose up -d
   # check 
   docker ps
   # should see kafka-kraft started
   ```
3. check if kafka is available (same as before)
   ```bash
   # enter kafka container
   docker exec -it kafka-kraft bash
   
   # create topic
   kafka-topics \
      --create \
      --topic kraft-topic \
      --bootstrap-server localhost:9092 \
      --partitions 1 \
      --replication-factor 1
   
   # produce topic
   kafka-console-producer \
      --topic kraft-topic \
      --bootstrap-server localhost:9092
   # input:
   hello
   kraft

   # consume topic (in another terminal)
   kafka-console-consumer \
      --topic kraft-topic \
      --from-beginning \
      --bootstrap-server localhost:9092
   # if see hello kraft, kafka runs successfully
   ```



## Client library 
```bash
# kafka-python has stopped maintenance, and support python 3.10+, so use:
pip install kafka-python-ng
# or a production ready one
pip install confluent-kafka
```

<img src='./pic/4_kafka_client.png' width=400>

key characteristics of Kafka client libraries:

- **Thick/Smart Client Architecture**:
  - Kafka clients are indeed thick clients with significant responsibilities beyond simple message passing
  - They **maintain local metadata** about the **cluster topology, broker locations, partition leaders, and consumer group coordination**
  - This contrasts with thin clients that might just send requests to a central server
- **High Availability & Resilience:**

  - The client handles **automatic retries, backpressure, and failover logic**
  - It maintains **connections to multiple brokers** and can **route requests to the appropriate partition leaders**
  - Implements producer acknowledgment strategies (acks=0,1,all) for durability guarantees

- **Exactly-Once Semantics**:
  
  - The client library manages **idempotent producers and transactional capabilities**
  - Handles complex coordination for exactly-once delivery when properly configured
  - This is indeed pushed down to the client rather than being purely broker-side

- **Offset Management**:

  - Automatic offset commits with configurable intervals
  - Handles the complexity of checkpointing consumer progress
  - Developers can focus on processing logic rather than manual offset tracking (though manual control is available when needed)
  
- **Battle-Tested Reliability**:
  - offset management is complex and error-prone-issues like duplicate processing, message loss, or consumer rebalancing complications are why using the library's built-in mechanisms is recommended

This applies to kafka-python-ng just as it does to other Kafka client libraries (like the Java client). The library abstracts away much of the distributed systems complexity while still exposing control when needed.